In [ ]:
from pathlib import Path

from tqdm import tqdm
from dotenv import dotenv_values
from IPython.display import Image

import httpx
import boto3
import xarray as xr
import geopandas as gpd
import pystac_client

In [ ]:
ENV = dotenv_values("../.env")
VALID_TILES = ["48MXU", "48MYU", "48MXT", "48MYT"]

## ESA Copernicus CDSE

In [4]:
root_catalog = pystac_client.Client.open("https://stac.dataspace.copernicus.eu/v1")
root_catalog

<Client id=cdse-stac>

In [16]:
time_range = "2026-01-01/2026-04-20"
bbox = [106.4012, -6.7885, 107.2277, -6.3022]

search = root_catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=time_range,
    # filter_lang="cql2-json",
    # filter={"op": "like", "args": [{"property": "id"}, "S2A"]}
)

items = search.item_collection()
len(items)

93

In [17]:
df = gpd.GeoDataFrame.from_features(items.to_dict(), crs="epsg:4326")
df.head(2)

,geometry,gsd,created,expires,updated,_private,datetime,platform,grid:code,published,...,sat:relative_orbit,view:sun_elevation,processing:datetime,processing:facility,processing:software,eopf:instrument_mode,eopf:origin_datetime,view:incidence_angle,product:timeliness_category,sat:platform_international_designator
0,"POLYGON ((106.80473 -5.42553, 106.80795 -6.418...",10,2026-04-20T12:11:00.000000Z,2262-01-01T00:00:00.000000Z,2026-04-20T13:36:24.955232Z,"{'visible': True, 'product_name': 'S2A_MSIL2A_...",2026-04-20T03:01:51.024000Z,sentinel-2a,MGRS-48MYU,2026-04-20T13:36:24.955232Z,...,32,61.680245,2026-04-20T11:21:08.000000Z,ESA,{'eometadatatool': '260327122600+dirty'},INS-NOBS,2026-04-20T12:11:00.000000Z,5.537653,NRT,2015-028A
1,"POLYGON ((107.80392 -7.07261, 107.79957 -6.324...",10,2026-04-20T12:11:15.000000Z,2262-01-01T00:00:00.000000Z,2026-04-20T13:37:38.069975Z,"{'visible': True, 'product_name': 'S2A_MSIL2A_...",2026-04-20T03:01:51.024000Z,sentinel-2a,MGRS-48MYT,2026-04-20T13:37:38.069975Z,...,32,61.166614,2026-04-20T11:21:08.000000Z,ESA,{'eometadatatool': '260327122600+dirty'},INS-NOBS,2026-04-20T12:11:15.000000Z,7.143467,NRT,2015-028A


In [19]:
df["datetime"].value_counts().sort_index()

datetime
2026-01-03T03:00:19.024000Z    4
2026-01-08T03:01:21.025000Z    5
2026-01-13T02:59:59.024000Z    4
2026-01-18T03:00:41.025000Z    4
2026-01-23T02:59:19.024000Z    4
2026-01-28T03:00:01.025000Z    4
2026-02-02T02:58:29.024000Z    4
2026-02-12T02:57:29.024000Z    4
2026-02-17T02:58:01.025000Z    4
2026-02-22T02:56:29.024000Z    4
2026-02-27T02:57:01.024000Z    4
2026-03-04T02:55:19.024000Z    4
2026-03-09T02:55:31.025000Z    4
2026-03-14T02:55:39.024000Z    4
2026-03-19T02:55:31.024000Z    4
2026-03-24T02:55:39.024000Z    4
2026-03-29T02:55:51.025000Z    4
2026-03-31T03:02:01.024000Z    4
2026-04-03T02:55:19.024000Z    4
2026-04-08T02:55:41.025000Z    4
2026-04-13T02:55:19.024000Z    4
2026-04-18T02:55:41.025000Z    4
2026-04-20T03:01:51.024000Z    4
Name: count, dtype: int64

In [14]:
items[0]

<Item id=S2A_MSIL2A_20260420T030151_N0512_R032_T48MYU_20260420T112108>

In [13]:
Image(url=items[0].assets["thumbnail"].href, width=400)

In [15]:
items[0].assets["B02_10m"]

<Asset href=s3://eodata/Sentinel-2/MSI/L2A/2026/04/20/S2A_MSIL2A_20260420T030151_N0512_R032_T48MYU_20260420T112108.SAFE/GRANULE/L2A_T48MYU_A056543_20260420T031410/IMG_DATA/R10m/T48MYU_20260420T030151_B02_10m.jp2>

## Download using S3 boto3

In [31]:
DOWNLOAD_BUCKET = "eodata"
DOWNLOAD_ROOT_PATH = Path("/mnt/data/workspace/bogor-agrisat/data/sentinel-2")

In [29]:
session = boto3.session.Session()
s3 = boto3.client(
    "s3",
    endpoint_url="https://eodata.dataspace.copernicus.eu",
    aws_access_key_id=ENV["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=ENV["AWS_SECRET_ACCESS_KEY"],
    region_name="default",
)

In [27]:
object_key = items[0].assets["B02_10m"].href[12:]
object_key

'Sentinel-2/MSI/L2A/2026/04/20/S2A_MSIL2A_20260420T030151_N0512_R032_T48MYU_20260420T112108.SAFE/GRANULE/L2A_T48MYU_A056543_20260420T031410/IMG_DATA/R10m/T48MYU_20260420T030151_B02_10m.jp2'

In [24]:
save_path: Path = (
    DOWNLOAD_ROOT_PATH / items[0].assets["B02_10m"].extra_fields["file:local_path"]
)
save_path.parent.mkdir(parents=True, exist_ok=True)
save_path

PosixPath('/mnt/data/workspace/bogor-agrisat/data/sentinel-2/S2A_MSIL2A_20260420T030151_N0512_R032_T48MYU_20260420T112108.SAFE/GRANULE/L2A_T48MYU_A056543_20260420T031410/IMG_DATA/R10m/T48MYU_20260420T030151_B02_10m.jp2')

In [ ]:
response = s3.head_object(Bucket=DOWNLOAD_BUCKET, Key=object_key)
total_size = response["ContentLength"]

with tqdm(total=total_size, unit="B", unit_scale=True) as pbar:
    s3.download_file(
        Bucket=DOWNLOAD_BUCKET,
        Key=object_key,
        Filename=save_path,
        Callback=lambda bytes_transferred: pbar.update(bytes_transferred),
    )

Downloading: Blue (band 2) - 10m: 100%|██████████| 117M/117M [00:26<00:00, 4.43MB/s] 


In [42]:
save_path: Path = (
    DOWNLOAD_ROOT_PATH / items[0].assets["thumbnail"].extra_fields["file:local_path"]
)
save_path.parent.mkdir(parents=True, exist_ok=True)
save_path

PosixPath('/mnt/data/workspace/bogor-agrisat/data/sentinel-2/S2A_MSIL2A_20260420T030151_N0512_R032_T48MYU_20260420T112108.SAFE/S2A_MSIL2A_20260420T030151_N0512_R032_T48MYU_20260420T112108-ql.jpg')

In [45]:
with httpx.stream("GET", items[0].assets["thumbnail"].href) as response:
    total_size = int(response.headers.get("Content-Length", 0))

    with save_path.open("wb") as f:
        with tqdm(total=total_size, unit="B", unit_scale=True) as progress:
            for chunk in response.iter_bytes():
                f.write(chunk)
                progress.update(len(chunk))

100%|██████████| 162/162 [00:00<00:00, 317kB/s]


## Check Downloaded File

In [39]:
ds = xr.open_dataset(save_path, engine="rasterio")
ds

<xarray.Dataset> Size: 482MB
Dimensions:      (band: 1, x: 10980, y: 10980)
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 88kB 7e+05 7e+05 7e+05 ... 8.097e+05 8.098e+05
  * y            (y) float64 88kB 9.4e+06 9.4e+06 9.4e+06 ... 9.29e+06 9.29e+06
    spatial_ref  int64 8B ...
Data variables:
    band_data    (band, y, x) float32 482MB ...